# Next-Day AQI Forecasting: Feature Engineering and Modelling

**Goal:** predict tomorrow's AQI category (`tomorrow_status`, six classes from *Good* to *Severe*) for 14 Indian cities.

This notebook builds on `eda.ipynb` (decisions D1–D12 in `docs/EDA_Decision_Document.pdf`). Part 1 builds the model features and checks them; part 2 trains the baselines.

**Part 1, feature engineering**
1. Build the dataset with the EDA pipeline, in the order that avoids leakage
2. Handle the rows whose trend features are empty
3. Split by time
4. Transforms for logistic regression
5. Rerun the VIF check with the lag and rolling features
6. Final feature sets

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utility import (AQI_CATEGORIES, prepare_dataset, split_train_val_test,
                     add_lr_features, vif_table, class_signal_summary)

pd.set_option('display.max_columns', 40)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'axes.axisbelow': True})
COLOR = '#2a78d6'

DATA_PATH = '../data/processed/aqi_weather_2015_2025.csv'
TARGET = 'tomorrow_status'
TREND = ['aqi_roll7_mean', 'aqi_roll7_std', 'aqi_diff1', 'aqi_dev7']

## 2. Build the dataset

`prepare_dataset` (in `utility.py`) applies the EDA decisions in a fixed order. The order matters: two of the steps protect against leakage only in this sequence.

| Step | What | Why this position |
|---|---|---|
| 1 | Rename to `aqi_today` / `status_today` | Makes the time direction explicit |
| 2 | Pollutant flags `has_*` (NH<sub>3</sub> dropped) | D5 |
| 3 | `tomorrow_status` from the next day's AQI **number**, per city | **Before filling**, so a filled value never becomes an answer (D1, D2) |
| 4 | `ffill_aqi(limit=3)`, adds `aqi_imputed` | Fills inputs only (D3) |
| 5 | `add_trend_features`: lag / rolling features + `month` | **Before dropping rows**, so a dropped day still counts as "yesterday" (D11) |
| 6 | Drop rows with no AQI today or no real tomorrow; drop the station count | D2, D3 |

The function reproduces the EDA notebook's data exactly (48,827 rows), plus the five new columns.

In [ ]:
df = prepare_dataset(DATA_PATH)
FLAGS = [c for c in df.columns if c.startswith('has_')]

WEATHER = ['temp_mean', 'temp_max', 'temp_min', 'temp_range', 'humidity_mean', 'dew_point_mean', 'precip_sum',
           'pressure_mean', 'cloud_cover_mean', 'wind_speed_mean', 'wind_speed_max', 'wind_u_mean', 'wind_v_mean',
           'wind_dir_mean']
groups = {'id': ['city', 'date'], 'AQI today': ['aqi_today', 'status_today', 'aqi_imputed'],
          'pollutant flags (D5)': FLAGS, 'weather': WEATHER, 'trend + calendar (D11)': TREND + ['month'],
          'target': [TARGET]}
assert sorted(sum(groups.values(), [])) == sorted(df.columns), 'every column belongs to one group'

print(f'Rows: {len(df):,} | columns: {df.shape[1]}')
pd.DataFrame({'columns': [', '.join(v) for v in groups.values()], 'count': [len(v) for v in groups.values()]},
             index=list(groups))

In [ ]:
# A few rows across the feature groups
df[['city', 'date', 'aqi_today'] + FLAGS + ['aqi_imputed', 'month'] + TREND + [TARGET]].head(10)

The first rows of each city have empty trend features: a 7-day window needs at least 4 known days, and the first day has no "yesterday". This is expected; section 3 handles it.

## 3. Rows with empty trend features

In [ ]:
empty = df[TREND].isna().any(axis=1)
print(f'Rows with at least one empty trend feature: {empty.sum()} ({empty.mean():.2%})')
print('By year:', df.loc[empty, 'date'].dt.year.value_counts().sort_index().to_dict())
pd.DataFrame({'all rows': df[TARGET].value_counts(normalize=True),
              'empty-trend rows': df.loc[empty, TARGET].value_counts(normalize=True)}).reindex(AQI_CATEGORIES).mul(100).round(1)

**Finding:** 195 rows (0.4%) have an empty trend feature, 181 of them in 2015–2018 (the first days of each city and the days after long gaps). Their class mix is close to the overall one, with slightly more Severe days (4.6% against 2.4%), but that is only about 9 days out of 1,178 Severe days, too few to change the balance.

**Decision:** drop them from every split. Logistic regression cannot take empty values, and keeping **one** dataset for all models means every model is scored on exactly the same days. (Tree models could handle the gaps, but 0.4% of rows is not worth two different evaluation sets.)

In [ ]:
df = df[~empty].reset_index(drop=True)

df_train, df_val, df_test = split_train_val_test(df)
pd.DataFrame({'rows': [len(df_train), len(df_val), len(df_test)],
              'years': ['2015–2022', '2023', '2024–2025']}, index=['train', 'val', 'test'])

## 4. Transforms for logistic regression

Logistic regression gives each feature **one weight** and adds them up, so it can only use a feature whose effect is roughly "more of X → more (or less) likely". Tree models split on thresholds and need none of this. Three EDA decisions turn into new columns (`add_lr_features`):

| New column | Problem it fixes | EDA decision |
|---|---|---|
| `pressure_rel` = pressure − the city's normal pressure | Raw pressure mostly says **which city** (altitude): Bengaluru's normal is far below Delhi's | D8 |
| `precip_log` = log(1 + rain) | Rain is extremely skewed: 0 mm on most days, up to 251 mm. A few huge values would dominate the weight | D8 |
| `month_sin`, `month_cos` | Month as a number (1–12) says December is "12 times" January, and puts them far apart. On a circle they are neighbours | D7 |

The city's normal pressure is computed on the **training years only**, so nothing from 2023–2025 leaks into the features.

Why each transform works, with reusable rules for other problems: [`docs/feature_transformation_notes.md`](../docs/feature_transformation_notes.md).

In [ ]:
city_pressure = df_train.groupby('city')['pressure_mean'].mean()     # training years only

df_train = add_lr_features(df_train, city_pressure)
df_val = add_lr_features(df_val, city_pressure)
df_test = add_lr_features(df_test, city_pressure)

new_vs_old = ['pressure_mean', 'pressure_rel', 'precip_sum', 'precip_log', 'month', 'month_cos', 'month_sin']
class_signal_summary(df_train, new_vs_old)[['rho', 'direction', 'medians', 'shape']]

In [ ]:
print('Skew of rain:  raw', round(df_train['precip_sum'].skew(), 2), '-> log', round(df_train['precip_log'].skew(), 2))
print('Raw month as a number, rho with the class order:',
      round(df_train['month'].rank().corr(df_train[TARGET].cat.codes.rank()), 2))

**Findings:**
- **`pressure_rel` reveals a real signal that raw pressure hid.** Raw pressure: rho 0.18. Pressure relative to the city's normal: **rho 0.52**, in a straight line. Once altitude is removed, the weather effect appears: *higher pressure than usual → calm, stagnant air → pollution builds up.*
- **`month` as a plain number has no linear signal at all (rho 0.00)**: air is bad in December *and* January, the two ends of the scale. `month_cos` (high in winter, low in the monsoon) reaches **rho 0.48**. `month_sin` is weak alone but separates autumn (October–November, crop burning) from spring, which share the same `month_cos`.
- **`precip_log`:** rho is unchanged (rank correlation ignores a log), but the skew drops from 7.8 to 1.6, so a few extreme rain days no longer dominate the weight.

**Decision:** logistic regression uses `pressure_rel`, `precip_log`, `month_sin` and `month_cos` instead of `pressure_mean`, `precip_sum` and `month`. Tree models keep the raw columns.

## 5. VIF with the lag and rolling features

The EDA's reduced set (D10) had every VIF below 10. Now the trend features are added. VIF asks, for each feature, *"how well can the other features predict this one?"*: about 1 is unique, 5–10 moderate, above 10 mostly a copy. Training years only.

### 5.1 All four trend features together

In [ ]:
reduced_weather = ['temp_min', 'temp_range', 'humidity_mean', 'precip_sum', 'pressure_mean',
                   'cloud_cover_mean', 'wind_speed_mean', 'wind_u_mean', 'wind_v_mean']

vif_table(df_train[['aqi_today'] + TREND + reduced_weather]).to_frame('VIF: aqi_today + all trend features')

**Finding:** three features explode: `aqi_today` (≈ 341), `aqi_roll7_mean` (≈ 286) and `aqi_dev7` (≈ 54). They are built from the same few days:

- `aqi_roll7_mean` = the average of **today and the 6 days before**
- `aqi_dev7` = **today** minus the average of **the 7 days before**

So any two of them almost exactly give the third. It is the same trap as `temp_max`, `temp_min` and `temp_range`. One of the three has to go.

### 5.2 Which one to drop?

In [ ]:
option_a = ['aqi_today', 'aqi_roll7_std', 'aqi_diff1', 'aqi_dev7']        # drop aqi_roll7_mean
option_b = ['aqi_roll7_mean', 'aqi_roll7_std', 'aqi_diff1', 'aqi_dev7']   # drop aqi_today

pd.concat({'A: drop aqi_roll7_mean': vif_table(df_train[option_a + reduced_weather]),
           'B: drop aqi_today': vif_table(df_train[option_b + reduced_weather])}, axis=1)

In [ ]:
df_train[['aqi_today'] + TREND].corr().round(2)

**Finding:** both options bring every VIF back below 10, and the AQI features themselves to about 3.5 or lower.

**Decision: option A, drop `aqi_roll7_mean`.**
- `aqi_today` is the strongest single feature (rho 0.88 against 0.85) and the most recent information.
- `aqi_roll7_mean` correlates 0.91 with `aqi_today`: it is mostly a smoothed copy of it.
- What the weekly mean adds beyond today is kept in other forms: `aqi_dev7` (how far today is from last week) and `aqi_roll7_std` (how jumpy the week was).
- Tree models keep all four trend features.

### 5.3 The full candidate set for logistic regression

The final check combines option A with the transforms from section 4.

In [ ]:
LR_NUMERIC = ['aqi_today', 'aqi_roll7_std', 'aqi_diff1', 'aqi_dev7',
              'temp_min', 'temp_range', 'humidity_mean', 'precip_log', 'pressure_rel',
              'cloud_cover_mean', 'wind_speed_mean', 'wind_u_mean', 'wind_v_mean',
              'month_sin', 'month_cos']
LR_NUMERIC_NO_HUMIDITY = [c for c in LR_NUMERIC if c != 'humidity_mean']

pd.concat({'candidate set': vif_table(df_train[LR_NUMERIC]),
           'without humidity_mean': vif_table(df_train[LR_NUMERIC_NO_HUMIDITY])}, axis=1)

In [ ]:
# Why do temp_min, pressure_rel and month_cos rise? They describe the same thing: winter.
df_train[['temp_min', 'pressure_rel', 'month_cos', 'temp_range', 'humidity_mean']].corr().round(2)

**Findings:**
- **Every VIF is below 10** in the candidate set. The highest are `temp_range` (≈ 8.1) and `temp_min` (≈ 7.5).
- **A "winter" group has formed:** cold nights (`temp_min`), higher-than-usual pressure (`pressure_rel`) and the winter months (`month_cos`) correlate at about ±0.8 with each other. Each still carries some information of its own, so all three stay, but their individual weights should be read with care.
- **Dropping `humidity_mean`** brings every VIF down to about 5.5 or lower (`temp_range` falls from 8.1 to 3.6), because humidity and `temp_range` move together (r = −0.75).
- The trend features stay low (1.5–3.3): they add information the weather does not have.

**Decisions:**
- **Baseline logistic regression:** `LR_NUMERIC` (15 numeric features), standardised, plus the categorical features below.
- **Variant to test on validation:** `LR_NUMERIC_NO_HUMIDITY`. Keep humidity only if the validation score drops without it.
- If coefficients flip sign between runs, the winter group is the first place to look.

## 6. Final feature sets

In [ ]:
LR_CATEGORICAL = ['city']                       # one-hot encoded in the model pipeline (D4)
LR_BINARY = FLAGS + ['aqi_imputed']

TREE_FEATURES = (['aqi_today'] + TREND +
                 ['temp_mean', 'temp_max', 'temp_min', 'temp_range', 'humidity_mean', 'dew_point_mean',
                  'precip_sum', 'pressure_mean', 'cloud_cover_mean', 'wind_speed_mean', 'wind_speed_max',
                  'wind_u_mean', 'wind_v_mean', 'month'] + FLAGS + ['aqi_imputed', 'city'])

pd.DataFrame({
    'logistic regression': [len(LR_NUMERIC), len(LR_BINARY), 'city (one-hot, 14)', 'yes', 'pressure_rel, precip_log, month_sin/cos'],
    'tree models': [len(TREE_FEATURES) - len(LR_BINARY) - 1, len(LR_BINARY), 'city (one-hot, 14)', 'no', 'none (raw columns)'],
}, index=['numeric features', '0/1 features', 'categorical', 'standardise', 'transforms'])

In [16]:
# Final sanity checks before modelling: every column any model will use, in every split
model_cols = sorted(set(LR_NUMERIC + LR_BINARY + LR_CATEGORICAL) | set(TREE_FEATURES))

for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    X = part[model_cols]
    assert X.notna().all().all(), f'{name}: missing values'
    assert np.isfinite(X.select_dtypes('number')).all().all(), f'{name}: infinite values'   # city is text
    assert part[TARGET].notna().all(), f'{name}: missing target'
print(f'No missing or infinite values in the {len(model_cols)} model columns of any split.')

# Every split must contain all six classes, with enough days to score them
counts = pd.DataFrame({n: p[TARGET].value_counts() for n, p in
                       [('train', df_train), ('val', df_val), ('test', df_test)]}).reindex(AQI_CATEGORIES)
assert (counts > 0).all().all(), 'a split is missing a class'
counts

No missing or infinite values in the 31 model columns of any split.


,train,val,test
tomorrow_status,,,
Good,2990,293,737
Satisfactory,10368,1785,3783
Moderate,9826,1829,3428
Poor,5615,778,1446
Very Poor,3696,368,521
Severe,1048,51,70


**Findings:**
- Every column used by any model is complete and finite in all three splits, and every row has a target.
- All six classes appear in every split, but **the worst classes become much rarer in later years:**

| Class | Train (2015–2022) | Validation (2023) | Test (2024–2025) |
|---|---|---|---|
| Very Poor | 11.0% | 7.2% | 5.2% |
| Severe | 3.1% (1,048 days) | **1.0% (51 days)** | **0.7% (70 days)** |

**What this means for modelling:**
- **Severe recall on validation is noisy.** With 51 Severe days, each missed day moves recall by about 2 percentage points. Compare models on Severe recall only when the gap is clearly larger than that, and read it together with macro F1.
- **The model will see more bad days in training than it meets later.** Trained on a 3.1% Severe rate, and with `class_weight='balanced'` pushing it further, it may predict Very Poor and Severe too often in 2023–2025. Check the **precision** of those classes (how many predicted Severe days really were Severe), not only recall.
- This is a shift over time (drift), worth a dedicated check before the final model: is air quality improving, or did some cities' worst periods drop out of the data?

## Summary of part 1

| Step | Decision |
|---|---|
| Dataset | `prepare_dataset`: EDA pipeline in leakage-safe order, 48,827 rows |
| Empty trend features | Drop 195 rows (0.4%) from every split, so all models are scored on the same days |
| Split | Train 2015–2022, validation 2023, test 2024–2025 |
| Transforms (logistic regression) | `pressure_rel` (rho 0.18 → 0.52), `precip_log` (skew 7.8 → 1.6), `month_sin` / `month_cos` (raw month rho 0.00 → 0.48) |
| Trend features | `aqi_today` + all four trend features is collinear (VIF ≈ 341): drop `aqi_roll7_mean` for logistic regression |
| Final LR set | 15 numeric + 7 binary + city; every VIF < 10; test dropping `humidity_mean` |
| Tree models | All raw features, all trend features, no transforms |

**Next (part 2):**
1. Persistence baseline: tomorrow's category = today's. This is the bar every model must beat.
2. Logistic regression baseline on `LR_NUMERIC` + `LR_BINARY` + one-hot `city`, standardised, `class_weight='balanced'`; scored with macro F1 and Very Poor / Severe recall on validation.
3. Variants: without `humidity_mean`, without `city`.